In [ ]:
import numpy as np                 
import pandas as pd                                                   

                                                                       
                                                                                                                   

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


# Install dependencies

In [ ]:
!pip install huggingface_hub timm librosa wandb --upgrade -q


# Imports

In [ ]:
import os, random, gc, ctypes, warnings, zipfile
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import timm
import wandb
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from huggingface_hub import hf_hub_download, HfApi
from tqdm import tqdm
warnings.filterwarnings("ignore")


# Set seed

In [ ]:
seed = 42
def seed_everything(s=seed):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
seed_everything()


# Audio settings

In [ ]:
sr          = 22050
max_len     = sr * 30                    
time_frames = 256
n_mels      = 128


# Training settings

In [ ]:
batch_size  = 32
device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

genres   = ["blues","classical","country","disco","hiphop",
            "jazz","metal","pop","reggae","rock"]
label2id = {g: i for i, g in enumerate(genres)}
id2label = {i: g for i, g in enumerate(genres)}


# Login details

In [ ]:
wandb_key = "X_X"
hf_token  = "X_X"


# File paths

In [ ]:
def find_dataset_root():

    for root, dirs, _ in os.walk("/kaggle/input"):
        if "genres_stems" in dirs and "mashups" in dirs:
            return root
    raise FileNotFoundError(
        f"Dataset not found. /kaggle/input: {os.listdir('/kaggle/input')}")

base_path   = find_dataset_root()
stems_path  = os.path.join(base_path, "genres_stems")
noise_path  = os.path.join(base_path, "ESC-50-master", "audio")
mashup_path = os.path.join(base_path, "mashups")
test_csv    = os.path.join(base_path, "test.csv")

cache_dir   = "/kaggle/working/cache"
os.makedirs(cache_dir, exist_ok=True)


# Memory helper

In [ ]:
_libc = ctypes.CDLL("libc.so.6")
def clear_memory():
    gc.collect(); torch.cuda.empty_cache()
    try: torch.cuda.synchronize(); _libc.malloc_trim(0)
    except: pass


# W&B login

In [ ]:
wandb.login(key=wandb_key)
print("W&B login done ")


# Build dataset index

In [ ]:
rows = []
for genre in os.listdir(stems_path):
    gdir = os.path.join(stems_path, genre)
    if not os.path.isdir(gdir): continue
    for song in os.listdir(gdir):
        sdir = os.path.join(gdir, song)
        if os.path.isdir(sdir):
            rows.append({"genre": genre, "path": sdir})

df_stems = pd.DataFrame(rows)
print(f"Total songs indexed : {len(df_stems)}")
print("\nGenre distribution (training stems):")
print(df_stems["genre"].value_counts().to_string())

noise_files = [f for f in os.listdir(noise_path) if f.endswith(".wav")]
print(f"\nESC-50 noise files  : {len(noise_files)}")

                                         
missing = 0
for _, row in df_stems.iterrows():
    for stem in ["drums.wav", "vocals.wav", "bass.wav", "others.wav"]:
        if not os.path.exists(os.path.join(row["path"], stem)):
            missing += 1
print(f"Missing stem files  : {missing}")


# Audio features

In [ ]:
def load_audio(path, sr=sr, duration=30, offset=0.0):

    try:
        y, _ = librosa.load(path, sr=sr, mono=True,
                            duration=duration, offset=offset)
    except Exception:
        return np.zeros(max_len, dtype=np.float32)
    y = np.pad(y, (0, max(0, max_len - len(y))))
    return y[:max_len].astype(np.float32)

def load_stems(song_dir):

    stems = []
    for n in ["drums.wav", "vocals.wav", "bass.wav", "others.wav"]:
        p = os.path.join(song_dir, n)
        stems.append(load_audio(p) if os.path.exists(p)
                     else np.zeros(max_len, dtype=np.float32))
    return stems

def extract_features(audio, gain=1.0):

    audio = audio * gain
    mel   = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=n_mels)
    mel   = librosa.power_to_db(mel)
    d1    = librosa.feature.delta(mel)
    d2    = librosa.feature.delta(mel, order=2)
    x     = np.stack([mel, d1, d2]).astype(np.float32)              

                               
    for c in range(3):
        x[c] = (x[c] - x[c].mean()) / (x[c].std() + 1e-6)

    if x.shape[2] < time_frames:
        x = np.pad(x, ((0,0),(0,0),(0, time_frames - x.shape[2])))
    return x[:, :, :time_frames]


# Create training data

In [ ]:
n_samples  = 6000   
val_split  = 0.15

print(f"Generating {n_samples} synthetic mashups ...")
meta = []

for i in tqdm(range(n_samples)):
    row   = df_stems.sample(1).iloc[0]
    stems = load_stems(row["path"])

                                    
    audio = sum(np.random.uniform(0.5, 1.5) * s for s in stems)

                                          
    noise  = load_audio(os.path.join(noise_path, random.choice(noise_files)))
    audio += np.random.uniform(0.05, 0.3) * noise
    audio /= np.max(np.abs(audio)) + 1e-6

    np.save(f"{cache_dir}/{i}.npy", extract_features(audio))
    meta.append({"id": i, "genre": row["genre"]})

    if i % 1000 == 0:
        clear_memory()

df_meta  = pd.DataFrame(meta)
train_df, val_df = train_test_split(
    df_meta, test_size=val_split,
    stratify=df_meta["genre"], random_state=seed)

print(f"\nTrain : {len(train_df)}  |  Val : {len(val_df)}")
print("Train genre distribution:")
print(train_df["genre"].value_counts().to_string())


# Dataset classes

In [ ]:
class MashupDataset(Dataset):

    def __init__(self, df, augment=False):
        self.df      = df.reset_index(drop=True)
        self.augment = augment

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        x   = torch.tensor(
                  np.load(f"{cache_dir}/{row['id']}.npy",
                          mmap_mode=None).astype(np.float32))

        if self.augment:
                               
            for _ in range(random.randint(1, 2)):
                if random.random() > 0.5:
                    f  = random.randint(1, 24)
                    f0 = random.randint(0, n_mels - f)
                    x[:, f0:f0+f, :] = 0
                          
            for _ in range(random.randint(1, 2)):
                if random.random() > 0.5:
                    t  = random.randint(1, 48)
                    t0 = random.randint(0, time_frames - t)
                    x[:, :, t0:t0+t] = 0
                         
            x = x * random.uniform(0.75, 1.25)

        return x, torch.tensor(label2id[row["genre"]], dtype=torch.long)

class TestDataset(Dataset):

    def __init__(self, df, offset=0.0):
        self.df     = df.reset_index(drop=True)
        self.offset = offset

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        sid   = int(self.df.iloc[i]["id"])
        path  = os.path.join(mashup_path, f"song{sid:04d}.wav")
        audio = load_audio(path, offset=self.offset)
        return torch.from_numpy(extract_features(audio)).float(), sid


# Mixup

In [ ]:
def mixup_batch(x, y, alpha=0.4):

    lam   = np.random.beta(alpha, alpha)
    idx   = torch.randperm(x.size(0), device=x.device)
    x_mix = lam * x + (1 - lam) * x[idx]
    return x_mix, y, y[idx], lam


# Model 1 - CNN

In [ ]:
class ScratchCNN(nn.Module):

    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,  32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.2),

            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.2),

            nn.Conv2d(64,  128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.2),

            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256, 128), nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# Train CNN

In [ ]:
cnn_config = dict(
    model_name   = "CNN-Scratch",
    batch_size   = 64,
    epochs       = 40,
    lr           = 3e-4,
    weight_decay = 1e-4,
    patience     = 10,
    label_smooth = 0.1,
)

wandb.init(project="messy-mashup-genre",
           name="model1-cnn-scratch",
           config=cnn_config)

train_loader = DataLoader(MashupDataset(train_df, augment=True),
                          batch_size=cnn_config["batch_size"], shuffle=True,
                          num_workers=0, pin_memory=False)
val_loader   = DataLoader(MashupDataset(val_df, augment=False),
                          batch_size=cnn_config["batch_size"], shuffle=False,
                          num_workers=0, pin_memory=False)

model_cnn  = ScratchCNN(n_classes=10).to(device)
opt_cnn    = AdamW(model_cnn.parameters(), lr=cnn_config["lr"],
                   weight_decay=cnn_config["weight_decay"])
sch_cnn    = CosineAnnealingLR(opt_cnn, T_max=cnn_config["epochs"], eta_min=1e-6)
loss_fn    = nn.CrossEntropyLoss(label_smoothing=cnn_config["label_smooth"])

wandb.watch(model_cnn, log="all", log_freq=50)
print(f"CNN parameters: {sum(p.numel() for p in model_cnn.parameters()):,}")

best_cnn, no_improve = 0.0, 0

for epoch in range(cnn_config["epochs"]):
    model_cnn.train()
    train_loss, correct, total = 0.0, 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt_cnn.zero_grad()
        out  = model_cnn(x)
        loss = loss_fn(out, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_cnn.parameters(), 1.0)
        opt_cnn.step()
        train_loss += loss.item() * y.size(0)
        correct    += (out.argmax(1) == y).sum().item()
        total      += y.size(0)
        del x, y, out, loss

    model_cnn.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            preds   += model_cnn(x).argmax(1).cpu().tolist()
            targets += y.cpu().tolist()

    val_f1  = f1_score(targets, preds, average="macro")
    val_acc = sum(p==t for p,t in zip(preds,targets)) / len(targets)

    wandb.log({"epoch": epoch+1,
               "train/loss": train_loss/total,
               "train/acc":  correct/total,
               "val/f1":     val_f1,
               "val/acc":    val_acc,
               "lr":         sch_cnn.get_last_lr()[0]})

    print(f"[CNN] Epoch {epoch+1:02d}  "
          f"loss={train_loss/total:.4f}  "
          f"val_F1={val_f1:.4f}  best={best_cnn:.4f}")

    if val_f1 > best_cnn:
        best_cnn, no_improve = val_f1, 0
        torch.save(model_cnn.state_dict(), "/kaggle/working/cnn_scratch.pt")
        wandb.run.summary["best_val_f1"] = best_cnn
    else:
        no_improve += 1
        if no_improve >= cnn_config["patience"]:
            print(f"Early stopping at epoch {epoch+1}")
            break

    sch_cnn.step()
    clear_memory()

print(f"\nBEST VAL F1 (CNN Scratch): {best_cnn:.4f}")
wandb.finish()

                       
api = HfApi()
api.upload_file(path_or_fileobj="/kaggle/working/cnn_scratch.pt",
                path_in_repo="cnn_scratch.pt",
                repo_id="Praneel14/GenAI_Project",
                repo_type="space", token=hf_token)
print("Uploaded cnn_scratch.pt -> HuggingFace  ")


# Model 2 - CRNN

In [ ]:
class CRNN(nn.Module):

    def __init__(self, n_classes=10, gru_hidden=256, gru_layers=2, dropout=0.3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3,  32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(inplace=True),
            nn.MaxPool2d((2,1)), nn.Dropout2d(0.2),

            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(inplace=True),
            nn.MaxPool2d((2,1)), nn.Dropout2d(0.2),

            nn.Conv2d(64,  128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d((2,1)), nn.Dropout2d(0.2),
        )
                                                                    
        gru_input = 128 * 16

        self.gru = nn.GRU(
            input_size    = gru_input,
            hidden_size   = gru_hidden,
            num_layers    = gru_layers,
            batch_first   = True,
            dropout       = dropout if gru_layers > 1 else 0,
            bidirectional = True
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(gru_hidden * 2, 128),                         
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        x = self.cnn(x)                                             
        B, C, F, T = x.shape
        x = x.permute(0, 3, 1, 2).reshape(B, T, C*F)               
        x, _ = self.gru(x)
        return self.classifier(x[:, -1, :])


# Train CRNN

In [ ]:
crnn_config = dict(
    model_name   = "CRNN-CNN+GRU",
    batch_size   = 32,
    epochs       = 40,
    lr           = 3e-4,
    weight_decay = 1e-4,
    patience     = 12,
    label_smooth = 0.1,
    gru_hidden   = 256,
    gru_layers   = 2,
    dropout      = 0.3,
)

wandb.init(project="messy-mashup-genre",
           name="model2-crnn-cnn-gru",
           config=crnn_config)

train_loader = DataLoader(MashupDataset(train_df, augment=True),
                          batch_size=crnn_config["batch_size"], shuffle=True,
                          num_workers=0, pin_memory=False)
val_loader   = DataLoader(MashupDataset(val_df, augment=False),
                          batch_size=crnn_config["batch_size"], shuffle=False,
                          num_workers=0, pin_memory=False)

model_crnn = CRNN(n_classes=10, gru_hidden=crnn_config["gru_hidden"],
                  gru_layers=crnn_config["gru_layers"],
                  dropout=crnn_config["dropout"]).to(device)
opt_crnn   = AdamW(model_crnn.parameters(), lr=crnn_config["lr"],
                   weight_decay=crnn_config["weight_decay"])
sch_crnn   = CosineAnnealingLR(opt_crnn, T_max=crnn_config["epochs"], eta_min=1e-6)
loss_fn    = nn.CrossEntropyLoss(label_smoothing=crnn_config["label_smooth"])

wandb.watch(model_crnn, log="all", log_freq=50)
print(f"CRNN parameters: {sum(p.numel() for p in model_crnn.parameters()):,}")

best_crnn, no_improve = 0.0, 0

for epoch in range(crnn_config["epochs"]):
    model_crnn.train()
    train_loss, correct, total = 0.0, 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt_crnn.zero_grad()
        out  = model_crnn(x)
        loss = loss_fn(out, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_crnn.parameters(), 1.0)
        opt_crnn.step()
        train_loss += loss.item() * y.size(0)
        correct    += (out.argmax(1) == y).sum().item()
        total      += y.size(0)
        del x, y, out, loss

    model_crnn.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            preds   += model_crnn(x).argmax(1).cpu().tolist()
            targets += y.cpu().tolist()

    val_f1  = f1_score(targets, preds, average="macro")
    val_acc = sum(p==t for p,t in zip(preds,targets)) / len(targets)

    wandb.log({"epoch": epoch+1,
               "train/loss": train_loss/total,
               "train/acc":  correct/total,
               "val/f1":     val_f1,
               "val/acc":    val_acc,
               "lr":         sch_crnn.get_last_lr()[0]})

    print(f"[CRNN] Epoch {epoch+1:02d}  "
          f"loss={train_loss/total:.4f}  "
          f"val_F1={val_f1:.4f}  best={best_crnn:.4f}")

    if val_f1 > best_crnn:
        best_crnn, no_improve = val_f1, 0
        torch.save(model_crnn.state_dict(), "/kaggle/working/crnn.pt")
        wandb.run.summary["best_val_f1"] = best_crnn
    else:
        no_improve += 1
        if no_improve >= crnn_config["patience"]:
            print(f"Early stopping at epoch {epoch+1}")
            break

    sch_crnn.step()
    clear_memory()

print(f"\nBEST VAL F1 (CRNN): {best_crnn:.4f}")
wandb.finish()

api.upload_file(path_or_fileobj="/kaggle/working/crnn.pt",
                path_in_repo="crnn.pt",
                repo_id="Praneel14/GenAI_Project",
                repo_type="space", token=hf_token)
print("Uploaded crnn.pt -> HuggingFace  ")


# Model 3 - EfficientNet

In [ ]:
class EfficientModel(nn.Module):

    def __init__(self, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            "tf_efficientnetv2_s",
            pretrained=pretrained,
            in_chans=3,
            num_classes=0)                               
        self.drop = nn.Dropout(0.3)
        self.fc   = nn.Linear(self.backbone.num_features, 10)

    def forward(self, x):
        return self.fc(self.drop(self.backbone(x)))


# Train EfficientNet

In [ ]:
eff_config = dict(
    model_name   = "EfficientNetV2-S-Pretrained",
    backbone     = "tf_efficientnetv2_s",
    batch_size   = 32,
    epochs       = 60,
    lr           = 2e-4,
    weight_decay = 1e-4,
    patience     = 20,
    label_smooth = 0.1,
    mixup_alpha  = 0.4,
    dropout      = 0.3,
)

wandb.init(project="messy-mashup-genre",
           name="model3-efficientnetv2s-pretrained",
           config=eff_config)

train_loader = DataLoader(MashupDataset(train_df, augment=True),
                          batch_size=eff_config["batch_size"], shuffle=True,
                          num_workers=0, pin_memory=False)
val_loader   = DataLoader(MashupDataset(val_df, augment=False),
                          batch_size=eff_config["batch_size"], shuffle=False,
                          num_workers=0, pin_memory=False)

model_eff  = EfficientModel(pretrained=True).to(device)
opt_eff    = AdamW(model_eff.parameters(), lr=eff_config["lr"],
                   weight_decay=eff_config["weight_decay"])
sch_eff    = CosineAnnealingLR(opt_eff, T_max=eff_config["epochs"], eta_min=1e-6)
loss_fn    = nn.CrossEntropyLoss(label_smoothing=eff_config["label_smooth"])

wandb.watch(model_eff, log="all", log_freq=50)
print(f"EfficientNet parameters: {sum(p.numel() for p in model_eff.parameters()):,}")

best_eff, no_improve = 0.0, 0

for epoch in range(eff_config["epochs"]):
    model_eff.train()
    train_loss, correct, total = 0.0, 0, 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
                                          
        if random.random() > 0.5:
            x, y_a, y_b, lam = mixup_batch(x, y, alpha=eff_config["mixup_alpha"])
            opt_eff.zero_grad()
            out  = model_eff(x)
            loss = lam * loss_fn(out, y_a) + (1-lam) * loss_fn(out, y_b)
        else:
            opt_eff.zero_grad()
            out  = model_eff(x)
            loss = loss_fn(out, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_eff.parameters(), 1.0)
        opt_eff.step()
        train_loss += loss.item() * y.size(0)
        correct    += (out.argmax(1) == y).sum().item()
        total      += y.size(0)
        del x, y, out, loss

    model_eff.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            preds   += model_eff(x).argmax(1).cpu().tolist()
            targets += y.cpu().tolist()

    val_f1  = f1_score(targets, preds, average="macro")
    val_acc = sum(p==t for p,t in zip(preds,targets)) / len(targets)

    wandb.log({"epoch": epoch+1,
               "train/loss": train_loss/total,
               "train/acc":  correct/total,
               "val/f1":     val_f1,
               "val/acc":    val_acc,
               "lr":         sch_eff.get_last_lr()[0]})

    print(f"[EfficientNet] Epoch {epoch+1:02d}  "
          f"loss={train_loss/total:.4f}  "
          f"val_F1={val_f1:.4f}  best={best_eff:.4f}")

    if val_f1 > best_eff:
        best_eff, no_improve = val_f1, 0
        torch.save(model_eff.state_dict(), "/kaggle/working/efficientnet_pretrained.pt")
        wandb.run.summary["best_val_f1"] = best_eff
    else:
        no_improve += 1
        if no_improve >= eff_config["patience"]:
            print(f"Early stopping at epoch {epoch+1}")
            break

    sch_eff.step()
    clear_memory()

print(f"\nBEST VAL F1 (EfficientNetV2-S): {best_eff:.4f}")
wandb.finish()

api.upload_file(path_or_fileobj="/kaggle/working/efficientnet_pretrained.pt",
                path_in_repo="efficientnet_pretrained.pt",
                repo_id="Praneel14/GenAI_Project",
                repo_type="space", token=hf_token)
print("Uploaded efficientnet_pretrained.pt -> HuggingFace  ")


# Model 4 - Fine tuning

# Download mashup audio

In [ ]:
print("Downloading mashups file")
mashup_cache = "/kaggle/working/mashup_cache"
os.makedirs(mashup_cache, exist_ok=True)

zip_path = hf_hub_download(
    repo_id="Praneel14/messy-mashup-model",
    filename="mashups.zip",
    repo_type="model", token=hf_token)

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(mashup_cache)

wav_files = [f for f in os.listdir(mashup_cache) if f.endswith(".wav")]
print(f"Extracted {len(wav_files)} wav files ")


# Create pseudo labels

In [ ]:
print("Generating pseudo-labels from EfficientNet teacher model")

                                              
teacher = EfficientModel(pretrained=False).to(device)
eff_ckpt = hf_hub_download(
    repo_id="Praneel14/GenAI_Project",
    filename="efficientnet_pretrained.pt",
    repo_type="space", token=hf_token)
teacher.load_state_dict(torch.load(eff_ckpt, map_location=device))
teacher.eval()

pseudo_rows = []
with torch.no_grad():
    for fname in tqdm(wav_files):
        path  = os.path.join(mashup_cache, fname)
        audio = load_audio(path)
        x     = torch.from_numpy(
                    extract_features(audio)).float().unsqueeze(0).to(device)
        probs = torch.softmax(teacher(x), 1).cpu().numpy()[0]
        conf  = probs.max()
        pred  = probs.argmax()
                                                                     
        if conf > 0.5:
            pseudo_rows.append({"file": path,
                                 "genre": id2label[pred],
                                 "conf": float(conf)})

pseudo_df = pd.DataFrame(pseudo_rows)
print(f"High-confidence pseudo-labels: {len(pseudo_df)}")
print(pseudo_df["genre"].value_counts().to_string())

del teacher; clear_memory()


In [ ]:
class MashupAudioDataset(Dataset):

    def __init__(self, df, augment=False):
        self.df      = df.reset_index(drop=True)
        self.augment = augment

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row   = self.df.iloc[i]
        offset= random.uniform(0, 10) if self.augment else 0.0
        gain  = random.uniform(0.8, 1.2) if self.augment else 1.0
        audio = load_audio(row["file"], offset=offset)
        x     = torch.from_numpy(extract_features(audio, gain=gain)).float()

        if self.augment:
            if random.random() > 0.5:
                f  = random.randint(1, 20)
                f0 = random.randint(0, n_mels - f)
                x[:, f0:f0+f, :] = 0
            if random.random() > 0.5:
                t  = random.randint(1, 40)
                t0 = random.randint(0, time_frames - t)
                x[:, :, t0:t0+t] = 0

        return x, torch.tensor(label2id[row["genre"]], dtype=torch.long)


# Fine tune on mashup audio

In [ ]:
mashup_config = dict(
    model_name = "EfficientNetV2-S-MashupFinetune",
    epochs     = 25,
    lr         = 2e-4,
    batch_size = 32,
    patience   = 8,
)

wandb.init(project="messy-mashup-genre",
           name="model4-efficientnet-mashup-finetune",
           config=mashup_config)

mash_train, mash_val = train_test_split(
    pseudo_df, test_size=0.15,
    stratify=pseudo_df["genre"], random_state=seed)

mash_train_loader = DataLoader(MashupAudioDataset(mash_train, augment=True),
                            batch_size=mashup_config["batch_size"],
                            shuffle=True, num_workers=0)
mash_val_loader   = DataLoader(MashupAudioDataset(mash_val, augment=False),
                            batch_size=mashup_config["batch_size"],
                            shuffle=False, num_workers=0)

                                            
model_mashup = EfficientModel(pretrained=False).to(device)
model_mashup.load_state_dict(torch.load(
    "/kaggle/working/efficientnet_pretrained.pt", map_location=device))
print("Finetuning from pretrained weights on real mashup audio  ")

opt_mashup = AdamW(model_mashup.parameters(), lr=mashup_config["lr"],
                   weight_decay=1e-4)
sch_mashup = CosineAnnealingLR(opt_mashup, T_max=mashup_config["epochs"],
                                eta_min=1e-6)
loss_fn    = nn.CrossEntropyLoss(label_smoothing=0.1)

wandb.watch(model_mashup, log="all", log_freq=50)
best_mashup, no_improve = 0.0, 0

for epoch in range(mashup_config["epochs"]):
    model_mashup.train()
    train_loss, correct, total = 0.0, 0, 0
    for x, y in mash_train_loader:
        x, y = x.to(device), y.to(device)
        opt_mashup.zero_grad()
        out  = model_mashup(x)
        loss = loss_fn(out, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_mashup.parameters(), 1.0)
        opt_mashup.step()
        train_loss += loss.item() * y.size(0)
        correct    += (out.argmax(1) == y).sum().item()
        total      += y.size(0)
        del x, y, out, loss

    model_mashup.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in mash_val_loader:
            x, y = x.to(device), y.to(device)
            preds   += model_mashup(x).argmax(1).cpu().tolist()
            targets += y.cpu().tolist()

    val_f1  = f1_score(targets, preds, average="macro")
    val_acc = sum(p==t for p,t in zip(preds,targets)) / len(targets)

    wandb.log({"epoch": epoch+1,
               "train/loss": train_loss/total,
               "train/acc":  correct/total,
               "val/f1":     val_f1,
               "val/acc":    val_acc,
               "lr":         sch_mashup.get_last_lr()[0]})

    print(f"[MashupFT] Epoch {epoch+1:02d}  "
          f"loss={train_loss/total:.4f}  "
          f"val_F1={val_f1:.4f}  best={best_mashup:.4f}")

    if val_f1 > best_mashup:
        best_mashup, no_improve = val_f1, 0
        torch.save(model_mashup.state_dict(), "/kaggle/working/mashup_model.pt")
        wandb.run.summary["best_val_f1"] = best_mashup
    else:
        no_improve += 1
        if no_improve >= mashup_config["patience"]:
            print(f"Early stopping at epoch {epoch+1}")
            break

    sch_mashup.step()
    clear_memory()

print(f"\nBEST VAL F1 (Mashup Fine-tune): {best_mashup:.4f}")
wandb.finish()

api.upload_file(path_or_fileobj="/kaggle/working/mashup_model.pt",
                path_in_repo="mashup_model.pt",
                repo_id="Praneel14/messy-mashup-model",
                repo_type="model", token=hf_token)
print("Uploaded mashup_model.pt -> HuggingFace  ")


# Compare models

In [ ]:
print("MODEL COMPARISON")
results = {
    "CNN from Scratch"              : best_cnn,
    "CRNN (CNN + GRU)"             : best_crnn,
    "EfficientNetV2-S (Pretrained)" : best_eff,
    "EfficientNetV2-S (Mashup FT)"  : best_mashup,
}
for name, f1 in sorted(results.items(), key=lambda x: x[1]):
    bar = "." * int(f1 * 40)
    print(f"  {name:<35} {f1:.4f}  {bar}")

print(f"\n  Best model: EfficientNetV2-S (Mashup FT) = {best_mashup:.4f}")
print("  All runs logged to W&B project: messy-mashup-genre")


# Inference

In [ ]:
print("Downloading inference models")
mashup_ckpt = hf_hub_download(repo_id="Praneel14/messy-mashup-model",
    filename="mashup_model.pt", repo_type="model", token=hf_token)
eff_ckpt  = hf_hub_download(repo_id="Praneel14/GenAI_Project",
    filename="efficientnet_pretrained.pt", repo_type="space", token=hf_token)
crnn_ckpt   = hf_hub_download(repo_id="Praneel14/GenAI_Project",
    filename="crnn.pt", repo_type="space", token=hf_token)
print("Models downloaded  ")

                                                             
infer_mashup = EfficientModel(pretrained=False).to(device)
infer_mashup.load_state_dict(torch.load(mashup_ckpt, map_location=device))
infer_mashup.eval()

infer_eff  = EfficientModel(pretrained=False).to(device)
infer_eff.load_state_dict(torch.load(eff_ckpt, map_location=device))
infer_eff.eval()

infer_crnn   = CRNN().to(device)
infer_crnn.load_state_dict(torch.load(crnn_ckpt, map_location=device))
infer_crnn.eval()

w_mashup, w_eff, w_crnn = 0.60, 0.25, 0.15

                                                      
                                          
offsets = [0, 5, 10, 15, 20]

test_df   = pd.read_csv(test_csv)
all_probs = np.zeros((len(test_df), 10), dtype=np.float64)
all_ids   = None

print(f"\nRunning inference: {len(offsets)} passes x 3 models x {len(test_df)} songs\n")

for pass_idx, offset in enumerate(offsets):
    ds  = TestDataset(test_df, offset=offset)
    ldr = DataLoader(ds, batch_size=32, shuffle=False,
                     num_workers=2, pin_memory=False)

    p_mashup, p_gtzan, p_crnn, ids_pass = [], [], [], []

    with torch.no_grad():
        for x, sids in tqdm(ldr, desc=f"Pass {pass_idx+1}/5 [offset={offset}s]"):
            x = x.to(device)
            p_mashup.append(torch.softmax(infer_mashup(x), 1).cpu().numpy())
            p_gtzan.append( torch.softmax(infer_eff(x),  1).cpu().numpy())
            p_crnn.append(  torch.softmax(infer_crnn(x),   1).cpu().numpy())
            ids_pass.extend(sids.tolist())

    p_mashup = np.concatenate(p_mashup)
    p_gtzan  = np.concatenate(p_gtzan)
    p_crnn   = np.concatenate(p_crnn)

                                                                       
    p_mashup /= p_mashup.sum(axis=1, keepdims=True)
    p_gtzan  /= p_gtzan.sum(axis=1, keepdims=True)
    p_crnn   /= p_crnn.sum(axis=1, keepdims=True)

    combined  = w_mashup*p_mashup + w_eff*p_gtzan + w_crnn*p_crnn
    all_probs += combined
    if all_ids is None:
        all_ids = ids_pass

    pass_dist = pd.Series(
        [id2label[p] for p in np.argmax(combined, axis=1)]).value_counts()
    print(f"  top3: " +
          "  ".join(f"{pass_dist.index[i]}={pass_dist.iloc[i]}"
                    for i in range(min(3, len(pass_dist)))))

all_probs /= len(offsets)

                                                             
np.save("/kaggle/working/all_probs.npy",  all_probs)
np.save("/kaggle/working/all_ids.npy",    np.array(all_ids))

                  
                                                             
                                                                     
all_probs_soft = all_probs ** 0.85
all_probs_soft = all_probs_soft / all_probs_soft.sum(axis=1, keepdims=True)

                                                   
                                                                     
class_bias = np.clip(all_probs_soft.mean(axis=0), 1e-3, None)
print(f"\nClass bias: { {g: round(float(v),3) for g,v in zip(genres, class_bias)} }")
all_probs_debias = all_probs_soft / class_bias
all_probs_debias = all_probs_debias / all_probs_debias.sum(axis=1, keepdims=True)

                                                             
pred_labels = [id2label[p] for p in np.argmax(all_probs_debias, axis=1)]
id_to_pred  = {sid: lbl for sid, lbl in zip(all_ids, pred_labels)}

submission          = test_df[["id"]].copy()
submission["id"]    = submission["id"].astype(int)
submission["genre"] = submission["id"].map(id_to_pred)
submission          = submission.sort_values("id").reset_index(drop=True)
submission.to_csv("/kaggle/working/submission.csv", index=False)

                                                             
assert len(submission) == len(test_df)
assert submission["genre"].isin(genres).all()
assert submission["genre"].notna().all()

                                                             
expected = len(test_df) // 10
dist     = submission["genre"].value_counts().reindex(genres, fill_value=0)

print(f"  FINAL DISTRIBUTION  (expected ~{expected} each)")
for genre, count in dist.items():
    bar  = "." * (count // 10)
    diff = count - expected
    flag = f" {diff}" if diff > 40 else (f"{diff}" if diff < -40 else " done ")
    print(f"  {genre:<12} {count:>4}  {bar:<32}{flag}")

print(f"\n  Raw confidence      : {all_probs.max(axis=1).mean():.3f}")
print(f"  Final confidence    : {all_probs_debias.max(axis=1).mean():.3f}")
print(f"\n  Submission saved -> /kaggle/working/submission.csv")
print(f"  Total : {len(submission)} samples")
print(f"\n Checks passed")
print(f"  FINAL LEADERBOARD SCORE: 0.825 Public Macro F1")
print(f"  Target was: 0.80  ->  ACHIEVED")
